In [1]:
import gensim
from  gensim.models import KeyedVectors
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
embbedigns_load = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin',binary = True)

<h1>Prueba si es que funciona el modelo </h1>
<p> deberia de salir Queen

In [3]:
similitud = embbedigns_load.most_similar(positive =["King","Women"],negative = ["Men"])
print(f"King is a Men like {similitud[0][0]} is a Women")

King is a Men like Queen is a Women


<h1>Lectura de archivo</h1>
<p> se lee el archivo .csv para poder trabajar con el

In [4]:
vuln = pd.read_csv("dataset_vuln_con_descripciones.csv")
vuln

,cve_id,published,has_kev,version,baseScore,baseSeverity,attackComplexity,privilegesRequired,userInteraction,scope,...,ap_type_is_Language,ap_type_is_Architecture,ap_prevalence_is_Often,ap_prevalence_is_Undetermined,ap_prevalence_is_Sometimes,ap_prevalence_is_Rarely,epss,percentile,cwe_description,cve_description
0,CVE-1999-0006,1998-07-14,0,3.1,9.8,4,0,0,0,0,...,0,0,1,0,0,0,0.08244,0.91900,"The product reads data past the end, or before...",Buffer overflow in POP servers based on BSD/Qu...
1,CVE-1999-0012,1998-02-06,0,3.1,7.0,3,1,0,0,0,...,0,0,0,1,0,0,0.00456,0.63139,This attack-focused weakness is caused by inco...,Some web servers under Microsoft Windows allow...
2,CVE-1999-0013,1998-01-22,0,3.1,8.4,3,0,0,0,0,...,0,0,0,1,0,0,0.00541,0.66829,The product transmits or stores authentication...,Stolen credentials from SSH clients via ssh-ag...
3,CVE-1999-0022,1996-07-03,0,3.1,7.8,3,0,1,0,0,...,0,0,1,0,0,0,0.00254,0.48541,"The product reads data past the end, or before...",Local user gains root privileges via buffer ov...
4,CVE-1999-0029,1997-07-16,0,3.1,8.4,3,0,0,0,0,...,0,0,1,0,0,0,0.00380,0.58803,"The product reads data past the end, or before...",root privileges via buffer overflow in ordist ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195891,CVE-2025-9990,2025-09-05,0,3.1,8.1,3,1,0,0,0,...,1,0,1,0,0,0,0.00343,0.56302,The PHP application receives input from an ups...,The WordPress Helpdesk Integration plugin for ...
195892,CVE-2025-9991,2025-09-30,0,3.1,8.1,3,1,0,0,0,...,1,0,1,0,0,0,0.00343,0.56302,The PHP application receives input from an ups...,The Tiny Bootstrap Elements Light plugin for W...
195893,CVE-2025-9992,2025-09-18,0,3.1,6.4,2,0,1,0,1,...,0,0,1,0,0,0,0.00043,0.13042,The product does not neutralize or incorrectly...,"The Ghost Kit – Page Builder Blocks, Motion Ef..."
195894,CVE-2025-9993,2025-09-30,0,3.1,8.1,3,1,0,0,0,...,1,0,1,0,0,0,0.00129,0.33182,The PHP application receives input from an ups...,The Bei Fen – WordPress Backup Plugin plugin f...


In [5]:
vuln_num = vuln.drop(columns=["cve_description","cwe_description"])

In [7]:
vuln_num_2020_2025= vuln_num[(vuln_num["published_year"]>=2020)].reset_index(drop=True)

In [9]:
vuln_num_2020_2025.to_csv("dataset_vuln_2020_2025_num.csv",index=False,encoding="utf-8")

In [5]:
mayor_2020_inicial=vuln[(vuln["published_year"]>=2020)].reset_index(drop=True)

<h1>Tokenizar el texto</h1>
<p>primero el texto se deja todo en minuscula y solo texto, sin puntaciones ni coma</p>
<p>luego este texto se separa en palabras</p>
<p>Posteriormente lo convierto a vector con el modelo precargado de google</p>

In [6]:
def tokenizar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-z\s]","",texto)
    return texto.split() 

In [7]:
prueba = "HolA. ¿Como Estas?, Bien y tu!$%#-}+´´"
prueba_tokenizada = tokenizar(prueba)

prueba_tokenizada

['hola', 'como', 'estas', 'bien', 'y', 'tu']

In [8]:
def texto_a_vector(texto,modelo,dim):
    tokens = tokenizar(texto)
    vectores = [modelo[t] for t in tokens if t in modelo.key_to_index]
    if len(vectores) == 0:
        return np.zeros(dim,dtype=np.float32)
    return np.mean(vectores,axis=0) #averange pooling

In [9]:
cve_embeddings = np.vstack(
    vuln["cve_description"].apply(lambda x: texto_a_vector(x,embbedigns_load,300)).values
)
cwe_embeddings = np.vstack(
    vuln["cwe_description"].apply(lambda x: texto_a_vector(x,embbedigns_load,300)).values
)

In [10]:
cve_embeddings_2020_2025 = np.vstack(
    mayor_2020_inicial["cve_description"].apply(lambda x: texto_a_vector(x,embbedigns_load,300)).values
)
cwe_embeddings_2020_2025 = np.vstack(
    mayor_2020_inicial["cwe_description"].apply(lambda x: texto_a_vector(x,embbedigns_load,300)).values
)

<h1>Unir todo en un solo DF para pasarlo a los modelos</h1>
<p>se convertira el cve_embeddings y el cwe_embeddings en df para luego concatenarlos en df original, eliminar los textos y dejar solo los vectores</p>

In [12]:
cve_emb_df = pd.DataFrame(cve_embeddings,columns=[f"cve_emb_{i}" for i in range(300)])
cwe_emb_df = pd.DataFrame(cwe_embeddings,columns=[f"cwe_emb_{i}" for i in range(300)])

vuln_final = pd.concat(
    [vuln.drop(columns=["cve_description","cwe_description"]).reset_index(drop=True),
     cve_emb_df,cwe_emb_df],axis=1
)

<h1>Exportar el dataset final en csv</h1>

In [25]:
vuln_final.to_csv("dataset_vuln_embeddings.csv",index=False,encoding="utf-8")

<h1>Solo Texto</h1>

In [12]:
texto = pd.concat(
    [vuln_final["has_kev"],cve_emb_df,cwe_emb_df],axis=1
    ).reset_index(drop=True)

In [30]:
texto.to_csv("dataset_cve_cwe_embeddings.csv",index=False,encoding="utf-8")

In [13]:
solo_cve = pd.concat(
    [vuln_final["has_kev"],cve_emb_df],axis=1
).reset_index(drop=True)

In [32]:
solo_cve.to_csv("dataset_cve_embeddings.csv",index=False,encoding="utf-8")

In [14]:
solo_cwe = pd.concat(
    [vuln_final["has_kev"],cwe_emb_df],axis=1
).reset_index(drop=True)

In [34]:
solo_cwe.to_csv("dataset_cwe_embeddings.csv",index=False,encoding="utf-8")

<h2>Muestro desde el 2020 hasta el año 2025</h2>

<p>La ultima publish date de el dataframe es el 05/11/2025 y el inicio es desde el 01/01/2020</p> 

In [13]:
mayor_2020=vuln_final[(vuln_final["published_year"]>=2020)].reset_index(drop=True)
mayor_2020

,cve_id,published,has_kev,version,baseScore,baseSeverity,attackComplexity,privilegesRequired,userInteraction,scope,...,cwe_emb_290,cwe_emb_291,cwe_emb_292,cwe_emb_293,cwe_emb_294,cwe_emb_295,cwe_emb_296,cwe_emb_297,cwe_emb_298,cwe_emb_299
0,CVE-1999-0199,2020-10-06,0,3.1,9.8,4,0,0,0,0,...,-0.088757,0.062128,-0.052144,0.003397,-0.004795,0.097781,0.012206,-0.013044,0.045046,-0.004463
1,CVE-2002-20001,2021-11-11,0,3.1,7.5,3,0,0,0,0,...,-0.079889,0.017911,-0.016943,0.056374,0.026987,0.080777,-0.010026,-0.048026,-0.055520,-0.057385
2,CVE-2002-20002,2025-01-02,0,3.1,5.4,2,1,0,0,1,...,-0.070732,0.082321,-0.080610,0.002647,-0.021059,0.047195,-0.009773,-0.046183,-0.028637,-0.000657
3,CVE-2002-2438,2021-05-18,0,3.1,7.5,3,0,0,0,0,...,-0.012585,0.060767,-0.048742,0.053595,-0.004846,0.033325,0.032391,-0.052588,0.051455,-0.008625
4,CVE-2003-20001,2025-04-01,0,3.1,5.6,2,1,0,0,0,...,-0.028706,-0.007545,-0.001841,0.027378,-0.042999,0.053493,-0.007896,-0.058969,0.024688,-0.066354
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149644,CVE-2025-9990,2025-09-05,0,3.1,8.1,3,1,0,0,0,...,-0.018044,0.001918,0.024690,0.061059,0.004715,0.059261,0.026810,-0.090760,-0.022306,0.019159
149645,CVE-2025-9991,2025-09-30,0,3.1,8.1,3,1,0,0,0,...,-0.018044,0.001918,0.024690,0.061059,0.004715,0.059261,0.026810,-0.090760,-0.022306,0.019159
149646,CVE-2025-9992,2025-09-18,0,3.1,6.4,2,0,1,0,1,...,-0.030351,-0.000984,-0.082192,0.069937,-0.037640,0.051863,0.012220,-0.065667,0.005923,-0.010531
149647,CVE-2025-9993,2025-09-30,0,3.1,8.1,3,1,0,0,0,...,-0.018044,0.001918,0.024690,0.061059,0.004715,0.059261,0.026810,-0.090760,-0.022306,0.019159


In [16]:
mayor_2020.to_csv("dataset_vuln_2020_2025_num+texto.csv")

<h2>Muestro desde el 2020 hasta el año 2025 solo texto</h2>

In [14]:
cve_emb_2020_2025_df = pd.DataFrame(cve_embeddings_2020_2025,columns=[f"cve_emb_{i}" for i in range(300)])
cwe_emb_2020_2025_df = pd.DataFrame(cwe_embeddings_2020_2025,columns=[f"cwe_emb_{i}" for i in range(300)])

vuln_final_2020_2025_texto = pd.concat(
    [mayor_2020_inicial["has_kev"],
     cve_emb_2020_2025_df,cwe_emb_2020_2025_df],axis=1
).reset_index(drop=True)

In [15]:
vuln_final_2020_2025_texto.to_csv("dataset_vuln_2020_2025_texto.csv",index=False,encoding="utf-8")

<h1>TF-IDF</h1>

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [19]:
tfidf_cve = TfidfVectorizer(max_features=300, stop_words="english")
tfidf_cwe = TfidfVectorizer(max_features=300, stop_words="english")

In [20]:
cve_tfidf = tfidf_cve.fit_transform(vuln["cve_description"])
cwe_tfidf = tfidf_cwe.fit_transform(vuln["cwe_description"])

In [25]:
cve_tfidf_2020 = tfidf_cve.fit_transform(mayor_2020_inicial["cve_description"])
cwe_tfidf_2020 = tfidf_cwe.fit_transform(mayor_2020_inicial["cwe_description"])

In [21]:
cve_tfidf_df = pd.DataFrame(
    cve_tfidf.toarray(),
    columns=[f"cve_tfidf_{w}" for w in tfidf_cve.get_feature_names_out()]
)
cwe_tfidf_df = pd.DataFrame(
    cwe_tfidf.toarray(),
    columns=[f"cwe_tfidf_{w}" for w in tfidf_cwe.get_feature_names_out()]
)

vuln_final_TF_IDF = pd.concat(
    [vuln.drop(columns=["cve_description", "cwe_description"]).reset_index(drop=True),
     cve_tfidf_df, cwe_tfidf_df],
    axis=1
)

In [26]:
cve_tfidf_2020_2025_df = pd.DataFrame(
    cve_tfidf_2020.toarray(),
    columns=[f"cve_tfidf_{w}" for w in tfidf_cve.get_feature_names_out()]
)
cwe_tfidf_2020_2025_df = pd.DataFrame(
    cwe_tfidf_2020.toarray(),
    columns=[f"cwe_tfidf_{w}" for w in tfidf_cwe.get_feature_names_out()]
)

In [23]:
TF_IDF_mayor_2020 = vuln_final_TF_IDF[(vuln_final["published_year"]>=2020)].reset_index(drop=True)
TF_IDF_mayor_2020


,cve_id,published,has_kev,version,baseScore,baseSeverity,attackComplexity,privilegesRequired,userInteraction,scope,...,cwe_tfidf_variables,cwe_tfidf_verify,cwe_tfidf_verifying,cwe_tfidf_way,cwe_tfidf_web,cwe_tfidf_window,cwe_tfidf_wraparound,cwe_tfidf_write,cwe_tfidf_writes,cwe_tfidf_xml
0,CVE-1999-0199,2020-10-06,0,3.1,9.8,4,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,CVE-2002-20001,2021-11-11,0,3.1,7.5,3,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,CVE-2002-20002,2025-01-02,0,3.1,5.4,2,1,0,0,1,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
3,CVE-2002-2438,2021-05-18,0,3.1,7.5,3,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
4,CVE-2003-20001,2025-04-01,0,3.1,5.6,2,1,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149644,CVE-2025-9990,2025-09-05,0,3.1,8.1,3,1,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
149645,CVE-2025-9991,2025-09-30,0,3.1,8.1,3,1,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
149646,CVE-2025-9992,2025-09-18,0,3.1,6.4,2,0,1,0,1,...,0.0,0.0,0.0,0.0,0.272565,0.0,0.0,0.0,0.0,0.0
149647,CVE-2025-9993,2025-09-30,0,3.1,8.1,3,1,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


In [24]:
TF_IDF_mayor_2020.to_csv("dataset_vuln_TF_IDF_2020_2025_num+texto.csv",index=False,encoding="utf-8")

In [27]:
vuln_final_2020_2025_texto_tf_idf = pd.concat(
    [mayor_2020_inicial["has_kev"],
     cve_tfidf_2020_2025_df,cwe_tfidf_2020_2025_df],axis=1
).reset_index(drop=True)

In [28]:
vuln_final_2020_2025_texto_tf_idf.to_csv("dataset_vuln_TF_IDF_2020_2025_texto.csv",index=False,encoding="utf-8")